In [2]:
import pandas as pd

In [3]:
df= pd.read_csv("Data/data.csv")

In [5]:
# 1. Ensure the column is actual datetime objects
df['TransactionStartTime'] = pd.to_datetime(df['TransactionStartTime'])

# 2. Now define the snapshot date
snapshot_date = df['TransactionStartTime'].max() + pd.Timedelta(days=1)

print(f"Snapshot Date: {snapshot_date}")

Snapshot Date: 2019-02-14 10:01:28+00:00


In [6]:
# 1. Calculate RFM
# Recency: Days since the snapshot date and the last transaction
# Frequency: Total number of transactions per customer
# Monetary: Total amount spent/transacted
rfm = df.groupby('CustomerId').agg({
    'TransactionStartTime': lambda x: (snapshot_date - x.max()).days,
    'TransactionId': 'count',
    'Amount': 'sum'
})

# Rename for clarity
rfm.columns = ['Recency', 'Frequency', 'Monetary']

# 2. Handle potential issues
# Ensure Monetary isn't negative for clustering (or handle according to business logic)
# K-Means works best with positive values for behavioral segmentation
rfm['Monetary'] = rfm['Monetary'].apply(lambda x: max(x, 0)) 

print(rfm.head())

                 Recency  Frequency  Monetary
CustomerId                                   
CustomerId_1          84          1       0.0
CustomerId_10         84          1       0.0
CustomerId_1001       90          5   20000.0
CustomerId_1002       26         11    4225.0
CustomerId_1003       12          6   20000.0


In [7]:
# 3. Pre-process (Scaling)
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm)

# 4. K-Means Clustering
# n_clusters=3 as per instructions; random_state for reproducibility
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

# 5. Analyze Cluster Profiles
cluster_analysis = rfm.groupby('Cluster').mean()
print("Cluster Average Metrics:")
print(cluster_analysis)

Cluster Average Metrics:
           Recency   Frequency      Monetary
Cluster                                     
0        61.870877    7.687719  6.180601e+04
1        12.727509   36.420848  1.802922e+05
2        27.200000  100.400000  5.069789e+07


In [8]:
# Identify the cluster ID based on your printed table
# Example: If Cluster 0 has the lowest Frequency and highest Recency:
risk_cluster_id = cluster_analysis['Frequency'].idxmin() 

# 6. Create the Target Variable
rfm['is_high_risk'] = (rfm['Cluster'] == risk_cluster_id).astype(int)

# 7. Merge back to the main processed dataset
# This is the 'df_final' or the main dataframe you'll use for training
df = df.merge(rfm[['is_high_risk']], on='CustomerId', how='left')

print("Target variable distribution:")
print(df['is_high_risk'].value_counts(normalize=True))

Target variable distribution:
is_high_risk
0    0.885482
1    0.114518
Name: proportion, dtype: float64


In [9]:
# Analyze the 3 clusters
cluster_stats = rfm.groupby('Cluster').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': 'mean'
}).reset_index()

print("Cluster Profiles:")
print(cluster_stats)

# Identify the High-Risk Cluster
# Logic: High Recency (hasn't visited in a long time) + Low Frequency + Low Monetary
# Replace 'X' with the Cluster index (0, 1, or 2) that fits this profile
risk_cluster_index = cluster_stats['Frequency'].idxmin() 
print(f"Identifying Cluster {risk_cluster_index} as the High-Risk Proxy.")

# 2. Assign the binary label
rfm['is_high_risk'] = (rfm['Cluster'] == risk_cluster_index).astype(int)

Cluster Profiles:
   Cluster    Recency   Frequency      Monetary
0        0  61.870877    7.687719  6.180601e+04
1        1  12.727509   36.420848  1.802922e+05
2        2  27.200000  100.400000  5.069789e+07
Identifying Cluster 0 as the High-Risk Proxy.


In [11]:
# Check the distribution using 'df' (or whatever you named your main dataframe)
print("High-Risk vs Low-Risk distribution:")
print(df['is_high_risk'].value_counts())

print("\nPercentage:")
print(df['is_high_risk'].value_counts(normalize=True))

High-Risk vs Low-Risk distribution:
is_high_risk
0    84707
1    10955
Name: count, dtype: int64

Percentage:
is_high_risk
0    0.885482
1    0.114518
Name: proportion, dtype: float64


In [12]:
# Ensure this line was executed successfully
df = df.merge(rfm[['is_high_risk']], on='CustomerId', how='left')

In [13]:
# Final step in your notebook
df.to_csv("Data/final_training_data.csv", index=False)